In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:

def get_wiki_links(page_url):
    """
    Fetches and extracts all hyperlinks from a given Wikipedia page URL.

    Args:
        page_url (str): The full URL of the Wikipedia page.

    Returns:
        dict: A dictionary containing three lists of links:
              'internal': Links to other Wikipedia articles.
              'external': Links to websites outside of Wikipedia.
              'references': Links to citations and references on the same page.
        Or None if the page cannot be fetched.
    """
    if not page_url.startswith("https://en.wikipedia.org/wiki/"):
        print("Error: Please provide a valid English Wikipedia article URL.")
        return None

    try:
        # Send a GET request to the URL
        response = requests.get(page_url)
        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return None

    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the main content area of the article to avoid pulling links from
    # sidebars, headers, etc. The main content is usually within a div with id="content".
    content_div = soup.find(id="content")
    if not content_div:
        print("Error: Could not find the main content div of the page.")
        return None

    # Initialize sets to store unique links
    internal_links = set()
    external_links = set()
    reference_links = set()

    # Find all anchor (<a>) tags within the content
    for a_tag in content_div.find_all('a', href=True):
        href = a_tag['href']

        # Case 1: Reference links (start with '#')
        if href.startswith('#'):
            # These are internal page links, often for citations.
            # We can create the full URL for clarity.
            full_url = urljoin(page_url, href)
            reference_links.add(full_url)

        # Case 2: Internal Wikipedia links (start with '/wiki/')
        elif href.startswith('/wiki/'):
            # Filter out non-article links like administrative pages or file pages
            if ':' not in href:
                full_url = urljoin("https://en.wikipedia.org", href)
                internal_links.add(full_url)

        # Case 3: External links (start with 'http' or 'https')
        elif href.startswith('http'):
            external_links.add(href)

    return {
        'internal': sorted(list(internal_links)),
        'external': sorted(list(external_links)),
        'references': sorted(list(reference_links))
    }

# --- Example Usage ---
if __name__ == "__main__":
    # Example Wikipedia page URL
    wiki_url = input()  #  "https://en.wikipedia.org/wiki/Python_(programming_language)"
    
    print(f"Attempting to pull links from: {wiki_url}\n")
    
    links = get_wiki_links(wiki_url)

    if links:
        print(f"--- Found {len(links['internal'])} Internal Links ---")
        for link in links['internal'][:10]: # Print first 10 as a sample
            print(link)
        if len(links['internal']) > 10:
            print("...")
        
        print(f"\n--- Found {len(links['external'])} External Links ---")
        for link in links['external'][:10]: # Print first 10 as a sample
            print(link)
        if len(links['external']) > 10:
            print("...")

        print(f"\n--- Found {len(links['references'])} Reference Links ---")
        for link in links['references'][:10]: # Print first 10 as a sample
            print(link)
        if len(links['references']) > 10:
            print("...")

 https://en.wikipedia.org/wiki/Python_(programming_language)


Attempting to pull links from: https://en.wikipedia.org/wiki/Python_(programming_language)

--- Found 748 Internal Links ---
https://en.wikipedia.org/wiki/%22Hello,_World!%22_program
https://en.wikipedia.org/wiki/3ds_Max
https://en.wikipedia.org/wiki/ABC_(programming_language)
https://en.wikipedia.org/wiki/ADMB
https://en.wikipedia.org/wiki/ALGOL
https://en.wikipedia.org/wiki/ALGOL_68
https://en.wikipedia.org/wiki/API
https://en.wikipedia.org/wiki/APL_(programming_language)
https://en.wikipedia.org/wiki/ATmega
https://en.wikipedia.org/wiki/AVR_microcontrollers
...

--- Found 596 External Links ---
http://archive.org/details/introducingpytho0000luba
http://boo.codehaus.org/Gotchas+for+Python+Users
http://cdsweb.cern.ch/journal/CERNBulletin/2006/31/News%20Articles/974627?ln=en
http://cobra-language.com/docs/acknowledgements/
http://doc.pypy.org/en/latest/stackless.html
http://download.tensorflow.org/paper/whitepaper2015.pdf
http://effbot.org/zone/call-by-object.htm
http://mypy-lang.org/


## 1. Google search-based retrievers

-    WebResearchRetriever: This retriever is designed for web search using Google Search API. It involves a multi-step process:
        Uses an LLM to generate multiple search questions based on the user's query.
        Executes these questions on the Google Search API to retrieve relevant web links.
        Scrapes and loads the content of the retrieved webpages into a vector store (e.g., Chroma, Pinecone) for further querying or summarization.
-    GoogleSearchAPIWrapper: This utility can be used in conjunction with other retrievers or agents to perform Google searches. It's used within the WebResearchRetriever. 

## 2. Other search APIs and web scraping tools

-    TavilySearchAPIRetriever: This retriever utilizes the Tavily Search API, which is specifically designed for LLM-optimized search results. According to a YouTube video discussing LangChain Agent Search Tools, Tavily helps to provide agents with up-to-date and relevant information.
-    SearchApiAPIWrapper: This utility integrates with the SearchApi Google Search API for real-time SERP scraping.
-    Bright Data (BrightDataWebScraperAPI): LangChain integrates with Bright Data's Web Scraper API, allowing you to scrape data from websites and use it to fuel your RAG applications.
-    Firecrawl: This open-source tool allows for efficient web scraping and is integrated with LangChain. A YouTube video demonstrates using Firecrawl with LangGraph (LangChain's agent framework) to create an agent that navigates websites, retrieves URLs, scrapes content, and evaluates the information. 

## 3. Retrievers for external indexes

-   ArxivRetriever: Allows querying the arXiv database for academic articles.
-   WikipediaRetriever: Allows users to query the Wikipedia database and fetch relevant articles.
-   AskNewsRetriever: This retriever integrates with AskNews, which provides a real-time news feed for LLMs.
-   PubMedRetriever: Provides access to the PubMed API for retrieving biomedical literature. 

# Dynamic promoting

In [ ]:
def select_prompt_based_on_query(user_query: str) -> str:
    """
    Selects a predefined prompt based on the user's input query.

    Args:
        user_query: The input query from the user.

    Returns:
        A selected prompt string or a default message if no match is found.
    """

    # Convert the user query to lowercase for case-insensitive matching
    normalized_query = user_query.lower()

    # Define a dictionary of keywords and their corresponding prompts
    # This is a simplified approach; in a real chat model, more advanced NLP
    # techniques like intent recognition would be used.
    predefined_prompts = {
        "greeting": "Hello! How can I assist you today?",
        "weather": "Please specify the location for the weather update. (e.g., 'What's the weather in London?')",
        "time": "What city or timezone are you interested in for the current time?",
        "help": "I can help with general questions, weather inquiries, or telling the current time. What do you need assistance with?",
        "joke": "Why don't scientists trust atoms? Because they make up everything!",
        "goodbye": "Goodbye! Have a great day!",
        # New, larger prompts
        "blog_writing": "Please provide the **topic** for the blog post you'd like me to write. What specific **key points** should I include?",
        "report_generation": "To generate a report, please specify the **topic** and any **specific sub-sections or data points** you need. Should it include **bullet points**? Do you require **reference URLs** for the information?",
        "code_generation": "What kind of **code** do you need? Please specify the **programming language**, the **functionality** required, and any **specific constraints or libraries**.",
        "summary_creation": "Please paste or provide a **link to the text** you'd like me to summarize. What's the **desired length** for the summary (e.g., a paragraph, bullet points, a few sentences)?"
    }

    # --- Simple keyword-based prompt selection logic ---

    if any(keyword in normalized_query for keyword in ["hi", "hello", "hey", "greetings"]):
        return predefined_prompts["greeting"]
    elif any(keyword in normalized_query for keyword in ["weather", "forecast", "climate"]):
        return predefined_prompts["weather"]
    elif any(keyword in normalized_query for keyword in ["time", "clock", "hour"]):
        return predefined_prompts["time"]
    elif any(keyword in normalized_query for keyword in ["help", "assist", "support"]):
        return predefined_prompts["help"]
    elif any(keyword in normalized_query for keyword in ["joke", "funny"]):
        return predefined_prompts["joke"]
    elif any(keyword in normalized_query for keyword in ["bye", "goodbye", "farewell"]):
        return predefined_prompts["goodbye"]
    # New keyword checks for larger prompts
    elif any(keyword in normalized_query for keyword in ["blog", "write blog", "create blog post"]):
        return predefined_prompts["blog_writing"]
    elif any(keyword in normalized_query for keyword in ["report", "generate report", "provide report"]):
        return predefined_prompts["report_generation"]
    elif any(keyword in normalized_query for keyword in ["code", "write code", "generate code", "program"]):
        return predefined_prompts["code_generation"]
    elif any(keyword in normalized_query for keyword in ["summarize", "summary", "condense"]):
        return predefined_prompts["summary_creation"]
    else:
        # Default prompt if no specific keywords are matched
        return "I'm sorry, I didn't quite understand that. Can you please rephrase or ask about something else?"

# --- Example Usage ---
if __name__ == "__main__":
    print("Welcome to the simple chat prompt selector!")
    print("Type 'quit' to exit.")

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'quit':
            break

        selected_prompt = select_prompt_based_on_query(user_input)
        print(f"Chat Model's Prompt: {selected_prompt}")



In [21]:
# URL based Loader
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [43]:
# Web based loader

import bs4
import os
from langchain_community.document_loaders import WebBaseLoader, SeleniumURLLoader
from selenium.common.exceptions import WebDriverException

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
import bs4
import os
from langchain_community.document_loaders import WebBaseLoader, SeleniumURLLoader
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from selenium.common.exceptions import WebDriverException
from bs4.element import Comment

# def get_important_content_from_url_offline_heuristic(url: str, ollama_model_name: str = "llama3") -> str:
    """
    Fetches, cleans, and summarizes content from a URL using a local LLM,
    without requiring a fixed CSS selector.

    Args:
        url (str): The URL of the webpage to process.
        ollama_model_name (str): The name of the Ollama model to use.

    Returns:
        str: A summary or the extracted content, or an error message.
    """
url: str = "https://python.langchain.com/docs/integrations/retrievers/cohere-reranker/#doing-reranking-with-coherererank"
ollama_model_name: str = "llama3"
    
# --- Step 1: Load content using fallback loaders ---
docs = None
try:
    # Use WebBaseLoader for a fast, initial attempt on static content
    loader = WebBaseLoader(web_paths=[url])
    docs = loader.load()
except Exception:
    print("WebBaseLoader failed, trying SeleniumURLLoader...")
    try:
        loader = SeleniumURLLoader(urls=[url])
        docs = loader.load()
    except WebDriverException as e:
        print(f"Selenium failed. Is your browser driver correctly installed? Error: {e}")
        #return f"Selenium failed. Is your browser driver correctly installed? Error: {e}"

docs = [i.page_content for i in docs]
if not docs:
    print("No content could be extracted from the URL.")
    #return "No content could be extracted from the URL."


# --- Step 3: Split cleaned content into chunks ---
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
all_splits = text_splitter.create_documents([text_content])

    # --- Step 4: Use the local LLM to get relevant content ---
    try:
        llm = Ollama(model=ollama_model_name)
        
        # Create a detailed prompt for the LLM to guide extraction
        prompt_template = PromptTemplate.from_template(
            """As an expert content analyst, your task is to extract the most important and relevant information from the following text.
            Focus on the core message, key facts, and main points. Do not include boilerplate text, navigation links, or irrelevant details.

            Source URL: {url}

            Text:
            {text}

            Most important and relevant content:
            """
        )

        final_summary = ""
        # Summarize each chunk to prevent exceeding the LLM context limit
        for chunk in all_splits:
            prompt = prompt_template.format(url=url, text=chunk.page_content)
            response = llm.invoke(prompt)
            final_summary += response.strip() + "\n\n"

        return final_summary
        
    except Exception as e:
        return f"An error occurred with the LLM or extraction chain. Is Ollama running? Error: {e}"


# --- Example Usage ---

# # Example with a blog post where the selector might change
# url_blog = "https://python.langchain.com/docs/integrations/retrievers/cohere-reranker/#doing-reranking-with-coherererank"
# print(f"Processing URL with heuristic approach: {url_blog}")
# extracted_content_heuristic = get_important_content_from_url_offline_heuristic(url_blog)
# print(extracted_content_heuristic[:1000]) # Print first 1000 characters


In [47]:
# --- Step 1: Load content using fallback loaders ---
docs = None
try:
    # Use WebBaseLoader for a fast, initial attempt on static content
    loader = WebBaseLoader(web_paths=[url])
    docs = loader.load()
except Exception:
    print("WebBaseLoader failed, trying SeleniumURLLoader...")

In [76]:
url: str = "https://python.langchain.com/docs/integrations/retrievers/cohere-reranker/#doing-reranking-with-coherererank"
ollama_model_name: str = "llama3"
    
docs = None
try:
    # Use WebBaseLoader for a fast, initial attempt on static content
    loader = WebBaseLoader(web_paths=[url])
    docs = loader.load()
except Exception:
    print("WebBaseLoader failed, trying SeleniumURLLoader...")
    try:
        loader = SeleniumURLLoader(urls=[url])
        docs = loader.load()
    except WebDriverException as e:
        print(f"Selenium failed. Is your browser driver correctly installed? Error: {e}")

if docs: # Check that docs is not None
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    # Correct method: split_documents() takes a list of Document objects
    all_splits = text_splitter.split_documents(docs)
    # Now all_splits is a list of new, smaller Document objects
    # You can access the content of each split document like this:
    # print(all_splits[0].page_content)
else:
    print("No documents were loaded.")


In [78]:
# --- Step 4: Use the local LLM to get relevant content ---
try:
    llm = Ollama(model=ollama_model_name)
    
    # Create a detailed prompt for the LLM to guide extraction
    prompt_template = PromptTemplate.from_template(
        """As an expert content analyst, your task is to extract the most important and relevant information from the following text.
        Focus on the core message, key facts, and main points. Do not include boilerplate text, navigation links, or irrelevant details.

        Source URL: {url}

        Text:
        {text}

        Most important and relevant content:
        """
    )

    final_summary = ""
    # Summarize each chunk to prevent exceeding the LLM context limit
    for chunk in all_splits:
        prompt = prompt_template.format(url=url, text=chunk.page_content)
        response = llm.invoke(prompt)
        final_summary += response.strip() + "\n\n"

    print(final_summary)
    
except Exception as e:
    print(f"An error occurred with the LLM or extraction chain. Is Ollama running? Error: {e}")


C:\Users\white\AppData\Local\Temp\ipykernel_15284\3829302821.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model=ollama_model_name)


An error occurred with the LLM or extraction chain. Is Ollama running? Error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002364F6D8040>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


# Break query into more dynamic prompt

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import List
load_dotenv() 
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [5]:
# Define a pydantic model to enforce the output structure
class Questions(BaseModel):
    questions: List[str] = Field( description="A list of sub-questions related to the input query." )

# Create an instance of the model and enforce the output structure
structured_model = model.with_structured_output(Questions)

# Define the system prompt
system = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered independently. \n"""

# Pass the question to the model
question = """What are the main components of an LLM-powered autonomous agent system?"""
result_question = structured_model.invoke([SystemMessage(content=system)]+[HumanMessage(content=question)]).content

for i in result_question.questions:
    print(i)

# Live information Pull:
- This example shows how to set up an agent that can perform a real-time web search using the open-source DuckDuckGo library.
- Set up Ollama with an open-source LLM:-   ollama pull llama3
- pip install langchain langchain-community langchain-core duckduckgo-search

In [80]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchRun, Tool
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# --- Step 1: Initialize the open-source LLM via Ollama ---
print("Initializing Hugging Face text generation model on GPU...")
llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_length=500,  # Adjust based on your needs
    device=device     # Use the configured device (GPU or CPU)
)

# Wrap the Hugging Face pipeline in a LangChain LLM object
llm = HuggingFacePipeline(pipeline=llm_pipeline)

# --- Step 2: Define the open-source tool for real-time data ---
# We use DuckDuckGoSearchRun to perform a real-time web search.
search = DuckDuckGoSearchRun()

# Convert the search functionality into a LangChain tool.
search_tool = Tool(
    name="duckduckgo_search",
    func=search._run,
    description="Useful for when you need to answer questions about current events or up-to-date information."
)

tools = [search_tool]

# --- Step 3: Initialize the Agent ---
# The LLM will use the tool description to decide when to call it.
agent_executor = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# --- Step 4: Run the agent with a real-time query ---
user_query = input()
print("\n--- Running Agent with Real-Time Query ---")
response = agent_executor.run(user_query)
print(f"\nAI Response: {response}")



KeyboardInterrupt



In [ ]:
# https://github.com/langchain-ai/rag-from-scratch/blob/main/rag_from_scratch_5_to_9.ipynb

# Text to SQl - Basic
- !pip install vanna vanna[sqlite]  --- its not working

In [8]:
# Creating SQL Lite DB and storing CSV file init
import sqlite3
import pandas as pd

# 1. Connect to the SQLite database.
conn = sqlite3.connect('Chinook.db')

df = pd.read_csv('C:/Users/white/OneDrive/Desktop/docs/Social_Network_Ads.csv')

# 3. Use the to_sql() method to write the DataFrame to a SQL table.
#    if_exists='replace' will drop the table if it already exists and recreate it.
#    index=False prevents pandas from adding a new index column to the table.
df.to_sql('products', conn, if_exists='replace', index=False)  # table name : 'products'
# here we can add another dataframe 'sales': df1.to_sql('sales', conn, if_exists='replace', index=False)

# 4. Close the connection.
conn.close()

In [9]:
# Text to SQL Query conversion
import os
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain.agents.agent_types import AgentType
from dotenv import load_dotenv
load_dotenv()  # LangChain will automatically pick up the GOOGLE_API_KEY from the environment

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite',temperature=0)

# Connect to the SQLite database
db = SQLDatabase.from_uri("sqlite:///Chinook.db")  # db = SQLDatabase.from_uri("Chinook.db")

In [10]:
# 4. Create the SQL agent
# This agent will use the Gemini LLM to decide on actions to take against the database.
agent_executor = create_sql_agent(
    llm= model,
    db=db,
    agent_type="tool-calling", # Use the GEMINI_FUNCTIONS agent type
    verbose=True
)

# 6. Query the database with natural language
question = "can you share data table shape and number of rows'?"
print(f"User question: {question}")

try:
    result = agent_executor.invoke({"input": question})
    print("\nQuery Result:")
    print(result['output'])
except Exception as e:
    print(f"An error occurred: {e}")


In [19]:
# Use SQLite DB to read tables and data:
db_file = "Chinook.db"
table_name = 'products'
sql_query = f"SELECT count(*) FROM {table_name}"

# Using python
try:
    with sqlite3.connect(db_file) as conn:
        df = pd.read_sql_query(sql_query, conn)
        print(f"Data from table '{table_name}':")
        print(df)

except sqlite3.Error as e:
    print(f"An error occurred: {e}")

Data from table 'products':
   count(*)
0       400


In [22]:
# Use SQLite DB to read tables and data:
db_file = "Chinook.db"
table_name = 'products'
sql_query = f"SELECT * FROM {table_name}"

# Using SQL
try:
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        records = cursor.fetchall()  # fetch all data from table

except sqlite3.Error as e:
    print(f"An error occurred: {e}")

In [25]:
# Code to fetch all existing tables in SQLite DB
db_file = "Chinook.db"
sql_query = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';" # Execute the query to get all user-defined tables

# Using python
try:
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        tables = cursor.fetchall()
        print(tables)

        if not tables:
            print("No user-defined tables found.")
        else:
            print("Existing tables:")
            for table in tables:
                print(table[0]) # The table name is the first element of each tuple


except sqlite3.Error as e:
    print(f"An error occurred: {e}")

[('products',)]


In [36]:
# Code to fetch all existing tables in SQLite DB
db_file = "Chinook.db"
table_name = 'products'
query = "SELECT sql FROM sqlite_master WHERE type='table' AND name=?;"
# Using python
try:
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(query, (table_name,))
        table = cursor.fetchall()
        print(table)

        if table:
            # The result is a tuple, e.g., ('CREATE TABLE ...',)
            table[0]
        else:
            print(f"Table '{table_name}' not found.")
            
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

[('CREATE TABLE "products" (\n"User ID" INTEGER,\n  "Gender" TEXT,\n  "Age" INTEGER,\n  "EstimatedSalary" INTEGER,\n  "Purchased" INTEGER,\n  "AD Type" TEXT\n)',)]


In [27]:
# Code to fetch table's schema in SQLite DB
db_file = "Chinook.db"
table_name = 'products'

# Using python
try:
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        # Execute PRAGMA to get column info
        cursor.execute(f"PRAGMA table_info({table_name});")

        # Fetch and print the results
        columns = cursor.fetchall()
        
        if not columns:
            print(f"Table '{table_name}' not found or has no columns.")
        else:
            print(f"Schema for table '{table_name}':")
            # Print headers
            print(f"{'cid':<3} | {'name':<15} | {'type':<10} | {'notnull':<7} | {'dflt_value':<12} | {'pk':<2}")
            print("-" * 59)
            # Print each row (column)
            for col in columns:
                print(f"{col[0]:<3} | {col[1]:<15} | {col[2]:<10} | {col[3]:<7} | {str(col[4]):<12} | {col[5]:<2}")
    

except sqlite3.Error as e:
    print(f"An error occurred: {e}")


Schema for table 'products':
cid | name            | type       | notnull | dflt_value   | pk
-----------------------------------------------------------
0   | User ID         | INTEGER    | 0       | None         | 0 
1   | Gender          | TEXT       | 0       | None         | 0 
2   | Age             | INTEGER    | 0       | None         | 0 
3   | EstimatedSalary | INTEGER    | 0       | None         | 0 
4   | Purchased       | INTEGER    | 0       | None         | 0 
5   | AD Type         | TEXT       | 0       | None         | 0 


In [7]:
# from langchain_community.utilities import SQLDatabase
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_community.agent_toolkits import create_sql_agent
# from langchain.agents.agent_types import AgentType